# MIRFLICKR-25K Demo - Fixed PySpark EDA, Metadata Extraction, and Memory-Style Analysis

This notebook is a full end-to-end version for your local MIRFLICKR-25K folder structure:

```text
/Users/dhruv/SJSU/sjsu/data 228 big data/dataset/mirflickr/
├── im1.jpg ... im25000.jpg
└── meta/
    ├── exif/
    ├── exif_raw/
    ├── license/
    ├── tags/
    └── tags_raw/
```

## What is fixed in this notebook

- Uses **image bytes** from Spark `binaryFile` to extract width/height correctly
- Uses a **MIRFLICKR-specific EXIF parser** for block format like:
  - `-Make`
  - `NIKON CORPORATION`
- Uses **safe array access** for `primary_location_tag`
- Uses **broadcast variables** for tag-based location extraction
- Includes **thorough EDA** and **10x10 image grids** filtered by location / bucket / memory cluster

## Main outputs

- `image_manifest.parquet`
- `mirflickr_features.parquet`
- `mirflickr_features.csv`
- charts saved under `output/charts/`

In [ ]:
# =========================
# 1. CONFIGURATION
# =========================

DATA_ROOT = "/Users/dhruv/SJSU/sjsu/data 228 big data/dataset/mirflickr"
IMAGE_ROOT = DATA_ROOT
META_ROOT = f"{DATA_ROOT}/meta"
OUTPUT_ROOT = f"{DATA_ROOT}/output"

SAMPLE_FRACTION = 1.0   # set to 0.1 while debugging if needed
RANDOM_SEED = 42

TOP_N_TAGS = 40
TOP_N_LOCATIONS = 30
TOP_N_CAMERAS = 20
TOP_N_LICENSES = 15

SAVE_CHARTS = True
MIN_PARTITIONS = 32

In [ ]:
# =========================
# 2. IMPORTS AND SPARK
# =========================

import os
import re
import io
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("MIRFLICKR-25K-Fixed-EDA")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

sc = spark.sparkContext
spark

In [ ]:
# =========================
# 3. HELPERS
# =========================

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
chart_dir = Path(OUTPUT_ROOT) / "charts"
chart_dir.mkdir(parents=True, exist_ok=True)

def save_current_chart(name):
    if SAVE_CHARTS:
        plt.tight_layout()
        plt.savefig(chart_dir / name, dpi=140, bbox_inches="tight")

def infer_photo_id_from_path(path_str):
    if path_str is None:
        return None
    fname = os.path.basename(path_str)
    stem = os.path.splitext(fname)[0]
    m = re.search(r"(\d+)", stem)
    return int(m.group(1)) if m else None

infer_photo_id_udf = F.udf(infer_photo_id_from_path, T.IntegerType())

def tokenize_tag_text(txt):
    if txt is None:
        return []
    txt = txt.replace(",", " ").replace(";", " ").replace("|", " ").replace("/", " ")
    txt = txt.replace("\\r\\n", " ").replace("\\n", " ").replace("\\r", " ").replace("\\t", " ")
    raw_tokens = re.split(r"\s+", txt.strip().lower())
    tokens = []
    for tok in raw_tokens:
        tok = re.sub(r"[^a-z0-9_\-]+", "", tok)
        if tok:
            tokens.append(tok)
    return tokens

tokenize_tag_text_udf = F.udf(tokenize_tag_text, T.ArrayType(T.StringType()))

@F.udf(T.TimestampType())
def parse_timestamp_udf(s):
    if s is None:
        return None
    s = str(s).strip()
    for fmt in [
        "%Y:%m:%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y/%m/%d %H:%M:%S",
        "%Y:%m:%d",
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%Y%m%d",
    ]:
        try:
            return pd.to_datetime(s, format=fmt).to_pydatetime()
        except Exception:
            pass
    try:
        ts = pd.to_datetime(s)
        return None if pd.isna(ts) else ts.to_pydatetime()
    except Exception:
        return None

@F.udf(T.DoubleType())
def safe_float_udf(s):
    if s is None:
        return None
    s = str(s).strip().replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    if not m:
        return None
    try:
        return float(m.group(0))
    except Exception:
        return None

def spark_file_uri_to_local_path(path_str):
    if path_str is None:
        return None
    if path_str.startswith("file:"):
        return path_str.replace("file:", "", 1)
    return path_str

In [ ]:
# =========================
# 4. LOAD IMAGE MANIFEST FROM TOP-LEVEL MIRFLICKR FOLDER
# =========================

image_df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.jpg")
    .option("recursiveFileLookup", "false")
    .load(IMAGE_ROOT)
    .select(
        F.col("path").alias("image_path"),
        F.col("length").alias("file_size_bytes"),
        F.col("modificationTime").alias("modification_time"),
        F.col("content")
    )
    .withColumn("photo_id", infer_photo_id_udf("image_path"))
)

if SAMPLE_FRACTION < 1.0:
    image_df = image_df.sample(False, SAMPLE_FRACTION, RANDOM_SEED)

print("Image count:", image_df.count())
image_df.select("photo_id", "image_path", "file_size_bytes").show(5, truncate=False)

In [ ]:
# =========================
# 5. IMAGE DIMENSIONS FROM BYTES (FIXED)
# =========================

@F.udf(
    T.StructType([
        T.StructField("width", T.IntegerType(), True),
        T.StructField("height", T.IntegerType(), True),
        T.StructField("format", T.StringType(), True),
        T.StructField("mode", T.StringType(), True),
    ])
)
def image_info_from_bytes_udf(content):
    try:
        if content is None:
            return (None, None, None, None)
        with Image.open(io.BytesIO(bytes(content))) as im:
            return (
                int(im.size[0]),
                int(im.size[1]),
                str(im.format),
                str(im.mode),
            )
    except Exception:
        return (None, None, None, None)

image_df = (
    image_df
    .withColumn("img_info", image_info_from_bytes_udf("content"))
    .withColumn("width", F.col("img_info.width"))
    .withColumn("height", F.col("img_info.height"))
    .withColumn("img_format", F.col("img_info.format"))
    .withColumn("img_mode", F.col("img_info.mode"))
    .drop("img_info", "content")
    .withColumn("aspect_ratio", F.when(F.col("height") > 0, F.col("width") / F.col("height")))
    .withColumn(
        "orientation",
        F.when(F.col("width").isNull() | F.col("height").isNull(), F.lit(None))
         .when(F.col("width") > F.col("height"), F.lit("landscape"))
         .when(F.col("width") < F.col("height"), F.lit("portrait"))
         .otherwise(F.lit("square"))
    )
)

image_df.select(
    F.count("*").alias("total"),
    F.sum(F.col("width").isNotNull().cast("int")).alias("width_non_null"),
    F.sum(F.col("height").isNotNull().cast("int")).alias("height_non_null")
).show()

image_df.select("photo_id", "width", "height", "aspect_ratio", "orientation").show(10, truncate=False)

In [ ]:
# =========================
# 6. TEXT METADATA LOADER
# =========================

def load_text_folder(folder_path, source_name):
    rdd = sc.wholeTextFiles(folder_path, minPartitions=MIN_PARTITIONS)
    if rdd.isEmpty():
        schema = T.StructType([
            T.StructField("meta_path", T.StringType(), True),
            T.StructField("raw_text", T.StringType(), True),
            T.StructField("source_name", T.StringType(), True),
        ])
        return spark.createDataFrame([], schema=schema)
    return (
        rdd.toDF(["meta_path", "raw_text"])
           .withColumn("source_name", F.lit(source_name))
    )

tags_df = load_text_folder(os.path.join(META_ROOT, "tags"), "tags")
tags_raw_df = load_text_folder(os.path.join(META_ROOT, "tags_raw"), "tags_raw")
exif_df = load_text_folder(os.path.join(META_ROOT, "exif"), "exif")
exif_raw_df = load_text_folder(os.path.join(META_ROOT, "exif_raw"), "exif_raw")
license_df = load_text_folder(os.path.join(META_ROOT, "license"), "license")

print("tags rows:", tags_df.count())
print("tags_raw rows:", tags_raw_df.count())
print("exif rows:", exif_df.count())
print("exif_raw rows:", exif_raw_df.count())
print("license rows:", license_df.count())

In [ ]:
# =========================
# 7. TAG PARSING
# =========================

tags_union_df = (
    tags_df.unionByName(tags_raw_df, allowMissingColumns=True)
    .withColumn("photo_id", infer_photo_id_udf("meta_path"))
    .withColumn("tag_list", tokenize_tag_text_udf("raw_text"))
    .withColumn("tag_count", F.size("tag_list"))
)

tag_exploded = (
    tags_union_df
    .select("photo_id", F.explode_outer("tag_list").alias("tag"))
    .where(F.col("tag").isNotNull() & (F.col("tag") != ""))
)

tags_by_photo = (
    tag_exploded
    .groupBy("photo_id")
    .agg(
        F.sort_array(F.collect_set("tag")).alias("tags"),
        F.count("tag").alias("tag_token_count")
    )
    .withColumn("unique_tag_count", F.size("tags"))
)

tags_by_photo.show(5, truncate=False)

In [ ]:
# =========================
# 8. MIRFLICKR EXIF PARSER (FIXED FOR BLOCK FORMAT)
# =========================

def parse_key_value_text(text):
    out = {}
    if not text:
        return out

    text = text.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\r", "\n")
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    i = 0

    while i < len(lines):
        line = lines[i]

        # MIRFLICKR EXIF keys start with "-"
        if line.startswith("-"):
            key = line[1:].strip()

            if i + 1 < len(lines) and not lines[i + 1].startswith("-"):
                value = lines[i + 1].strip()
                i += 2
            else:
                value = ""
                i += 1

            if key:
                out[key] = value
        else:
            i += 1

    return out

parse_kv_udf = F.udf(parse_key_value_text, T.MapType(T.StringType(), T.StringType()))

exif_parsed_df = (
    exif_df.unionByName(exif_raw_df, allowMissingColumns=True)
    .withColumn("photo_id", infer_photo_id_udf("meta_path"))
    .withColumn("kv", parse_kv_udf("raw_text"))
)

exif_parsed_df.select(
    "photo_id",
    F.size("kv").alias("n_keys"),
    F.map_keys("kv").alias("keys")
).show(5, truncate=False)

In [ ]:
# =========================
# 9. EXIF FEATURE EXTRACTION (FIXED KEY NAMES)
# =========================

def first_map_value_expr(keys):
    exprs = [F.element_at(F.col("kv"), F.lit(k)) for k in keys]
    return F.coalesce(*exprs)

exif_features_df = (
    exif_parsed_df
    .select(
        "photo_id",
        "source_name",
        first_map_value_expr([
            "Date and Time (Original)",
            "Date and Time (Digitized)",
            "Date and Time",
            "Date Created"
        ]).alias("raw_capture_ts"),
        first_map_value_expr(["Make"]).alias("camera_make"),
        first_map_value_expr(["Model"]).alias("camera_model"),
        first_map_value_expr(["GPS Latitude"]).alias("raw_gps_lat"),
        first_map_value_expr(["GPS Longitude"]).alias("raw_gps_lon"),
        first_map_value_expr(["Pixel X-Dimension", "Image Width"]).alias("raw_exif_width"),
        first_map_value_expr(["Pixel Y-Dimension", "Image Height"]).alias("raw_exif_height"),
        first_map_value_expr(["ISO Speed"]).alias("iso"),
        first_map_value_expr(["Exposure"]).alias("exposure_time"),
        first_map_value_expr(["Aperture"]).alias("f_number"),
        first_map_value_expr(["Focal Length"]).alias("focal_length")
    )
    .withColumn("capture_ts", parse_timestamp_udf("raw_capture_ts"))
    .withColumn("gps_lat", safe_float_udf("raw_gps_lat"))
    .withColumn("gps_lon", safe_float_udf("raw_gps_lon"))
    .withColumn("exif_width", safe_float_udf("raw_exif_width").cast("int"))
    .withColumn("exif_height", safe_float_udf("raw_exif_height").cast("int"))
)

exif_by_photo = (
    exif_features_df
    .groupBy("photo_id")
    .agg(
        F.first("capture_ts", ignorenulls=True).alias("capture_ts"),
        F.first("camera_make", ignorenulls=True).alias("camera_make"),
        F.first("camera_model", ignorenulls=True).alias("camera_model"),
        F.first("gps_lat", ignorenulls=True).alias("gps_lat"),
        F.first("gps_lon", ignorenulls=True).alias("gps_lon"),
        F.first("exif_width", ignorenulls=True).alias("exif_width"),
        F.first("exif_height", ignorenulls=True).alias("exif_height"),
        F.first("iso", ignorenulls=True).alias("iso"),
        F.first("exposure_time", ignorenulls=True).alias("exposure_time"),
        F.first("f_number", ignorenulls=True).alias("f_number"),
        F.first("focal_length", ignorenulls=True).alias("focal_length")
    )
)

exif_by_photo.show(5, truncate=False)

In [ ]:
# =========================
# 10. LICENSE PARSING
# =========================

license_parsed_df = (
    license_df
    .withColumn("photo_id", infer_photo_id_udf("meta_path"))
    .withColumn("license_text", F.trim(F.col("raw_text")))
)

license_by_photo = (
    license_parsed_df
    .groupBy("photo_id")
    .agg(F.first("license_text", ignorenulls=True).alias("license_text"))
)

license_by_photo.show(5, truncate=False)

In [ ]:
# =========================
# 11. JOIN EVERYTHING INTO ONE FEATURE TABLE
# =========================

features_df = (
    image_df
    .join(tags_by_photo, on="photo_id", how="left")
    .join(exif_by_photo, on="photo_id", how="left")
    .join(license_by_photo, on="photo_id", how="left")
    .withColumn("has_tags", F.col("tags").isNotNull())
    .withColumn("has_exif_capture_ts", F.col("capture_ts").isNotNull())
    .withColumn("has_gps_exif", F.col("gps_lat").isNotNull() & F.col("gps_lon").isNotNull())
    .withColumn("year", F.year("capture_ts"))
    .withColumn("month", F.month("capture_ts"))
    .withColumn("day", F.dayofmonth("capture_ts"))
)

features_df = features_df.cache()
_ = features_df.count()

features_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("width").isNotNull().cast("int")).alias("width_ok"),
    F.sum(F.col("height").isNotNull().cast("int")).alias("height_ok"),
    F.sum(F.col("capture_ts").isNotNull().cast("int")).alias("capture_ts_ok"),
    F.sum(F.col("camera_make").isNotNull().cast("int")).alias("camera_make_ok"),
    F.sum(F.col("camera_model").isNotNull().cast("int")).alias("camera_model_ok"),
    F.sum(F.col("has_tags").cast("int")).alias("has_tags_ok")
).show()

features_df.show(5, truncate=False)

In [ ]:
# =========================
# 12. LOCATION EXTRACTION FROM TAGS (SAFE + BROADCAST)
# =========================

countries = {
    "usa","us","unitedstates","united_states","canada","mexico","brazil","argentina","peru",
    "uk","unitedkingdom","united_kingdom","england","scotland","wales","ireland",
    "france","germany","italy","spain","portugal","netherlands","belgium","switzerland","austria",
    "sweden","norway","finland","denmark","poland","greece","turkey","russia","ukraine",
    "india","china","japan","korea","thailand","vietnam","indonesia","singapore","malaysia",
    "australia","newzealand","egypt","morocco","southafrica","kenya","uae","dubai"
}

major_places = {
    "newyork":"New York",
    "new_york":"New York",
    "nyc":"New York",
    "losangeles":"Los Angeles",
    "los_angeles":"Los Angeles",
    "sanfrancisco":"San Francisco",
    "san_francisco":"San Francisco",
    "london":"London",
    "paris":"Paris",
    "rome":"Rome",
    "milan":"Milan",
    "venice":"Venice",
    "berlin":"Berlin",
    "tokyo":"Tokyo",
    "kyoto":"Kyoto",
    "osaka":"Osaka",
    "beijing":"Beijing",
    "shanghai":"Shanghai",
    "hongkong":"Hong Kong",
    "hong_kong":"Hong Kong",
    "singapore":"Singapore",
    "sydney":"Sydney",
    "melbourne":"Melbourne",
    "toronto":"Toronto",
    "vancouver":"Vancouver",
    "seattle":"Seattle",
    "boston":"Boston",
    "chicago":"Chicago",
    "miami":"Miami",
    "dallas":"Dallas",
    "austin":"Austin",
    "washington":"Washington",
    "washingtondc":"Washington DC",
    "washington_dc":"Washington DC",
    "lasvegas":"Las Vegas",
    "las_vegas":"Las Vegas",
    "yosemite":"Yosemite",
    "california":"California",
    "texas":"Texas",
    "florida":"Florida",
    "arizona":"Arizona",
    "colorado":"Colorado",
    "hawaii":"Hawaii",
    "alaska":"Alaska",
    "japan":"Japan",
    "italy":"Italy",
    "france":"France",
    "germany":"Germany",
    "spain":"Spain",
    "india":"India",
    "canada":"Canada",
    "usa":"USA"
}

multiword_patterns = {
    "new york": "New York",
    "los angeles": "Los Angeles",
    "san francisco": "San Francisco",
    "hong kong": "Hong Kong",
    "las vegas": "Las Vegas",
    "united states": "USA",
    "united kingdom": "United Kingdom",
    "golden gate": "San Francisco"
}

countries_bc = sc.broadcast(countries)
major_places_bc = sc.broadcast(major_places)
multiword_patterns_bc = sc.broadcast(multiword_patterns)

@F.udf(T.ArrayType(T.StringType()))
def extract_location_candidates(tags):
    countries = countries_bc.value
    major_places = major_places_bc.value
    multiword_patterns = multiword_patterns_bc.value

    if not tags:
        return []

    cleaned = []
    for t in tags:
        if t is None:
            continue
        s = str(t).strip().lower().replace("-", "_")
        if s:
            cleaned.append(s)

    joined = " ".join([c.replace("_", " ") for c in cleaned])
    found = []

    for phrase, norm in multiword_patterns.items():
        if phrase in joined:
            found.append(norm)

    for t in cleaned:
        token = t.replace(" ", "_")
        token_flat = token.replace("_", "")
        if token in major_places:
            found.append(major_places[token])
        elif token_flat in major_places:
            found.append(major_places[token_flat])
        elif token in countries:
            found.append(token.replace("_", " ").title())
        elif token_flat in countries:
            found.append(token_flat.title())

    seen = set()
    out = []
    for x in found:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

features_df = (
    features_df
    .withColumn("location_candidates", extract_location_candidates("tags"))
    .withColumn(
        "primary_location_tag",
        F.when(
            F.col("location_candidates").isNotNull() & (F.size("location_candidates") > 0),
            F.col("location_candidates")[0]
        ).otherwise(F.lit(None))
    )
    .withColumn("has_location_tag", F.size("location_candidates") > 0)
)

features_df = features_df.cache()
_ = features_df.count()

features_df.select("photo_id", "tags", "location_candidates", "primary_location_tag").where(F.col("has_location_tag")).show(10, truncate=False)

In [ ]:
# =========================
# 13. SEMANTIC BUCKETS FROM TAGS
# =========================

portrait_terms = {"portrait","people","person","face","girl","boy","woman","man","baby","child","kids","bride"}
landscape_terms = {"landscape","nature","mountain","sea","beach","river","lake","sunset","sunrise","sky","forest","clouds","snow","waterfall","tree","trees"}
urban_terms = {"city","urban","street","architecture","building","buildings","bridge","road","downtown","tower","skyscraper","traffic"}
animal_terms = {"dog","cat","bird","horse","zoo","animal","animals","wildlife"}
food_terms = {"food","drink","coffee","tea","cake","dessert","pizza","restaurant"}
travel_event_terms = {"vacation","travel","trip","holiday","festival","wedding","party","concert","museum","park"}

@F.udf(T.StringType())
def infer_semantic_bucket(tags):
    if not tags:
        return "unknown"
    s = set([str(t).lower() for t in tags])
    if s & portrait_terms:
        return "portrait_people"
    if s & landscape_terms:
        return "landscape_nature"
    if s & urban_terms:
        return "urban_street"
    if s & animal_terms:
        return "animals"
    if s & food_terms:
        return "food"
    if s & travel_event_terms:
        return "travel_event"
    return "other"

features_df = features_df.withColumn("semantic_bucket", infer_semantic_bucket("tags"))
features_df = features_df.cache()
_ = features_df.count()

features_df.groupBy("semantic_bucket").count().orderBy(F.desc("count")).show()

In [ ]:
# =========================
# 14. SAVE CORE OUTPUTS
# =========================

manifest_out = os.path.join(OUTPUT_ROOT, "image_manifest.parquet")
features_out = os.path.join(OUTPUT_ROOT, "mirflickr_features.parquet")
features_csv_out = os.path.join(OUTPUT_ROOT, "mirflickr_features.csv")

image_df.write.mode("overwrite").parquet(manifest_out)
features_df.write.mode("overwrite").parquet(features_out)

features_csv_df = (
    features_df
    .withColumn("tags_str", F.concat_ws("|", F.col("tags")))
    .withColumn("location_candidates_str", F.concat_ws("|", F.col("location_candidates")))
    .drop("tags", "location_candidates")
)

features_csv_df.toPandas().to_csv(features_csv_out, index=False)

print("Saved:", manifest_out)
print("Saved:", features_out)
print("Saved:", features_csv_out)

## EDA Section
The next cells provide a thorough EDA for your demo and presentation.

In [ ]:
# =========================
# 15. BASIC OVERVIEW
# =========================

print("Row count:", features_df.count())
print("Column count:", len(features_df.columns))
print("Columns:")
for c in features_df.columns:
    print(" -", c)

features_df.printSchema()
features_df.show(5, truncate=False)

In [ ]:
# =========================
# 16. MISSINGNESS SUMMARY
# =========================

total_rows = features_df.count()

missing_exprs = [
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in features_df.columns
]

missing_pd = features_df.select(*missing_exprs).toPandas().T.reset_index()
missing_pd.columns = ["column", "null_count"]
missing_pd["null_pct"] = (missing_pd["null_count"] / total_rows * 100).round(2)
missing_pd = missing_pd.sort_values(["null_pct", "null_count"], ascending=False)

missing_pd.head(30)

In [ ]:
plt.figure(figsize=(12, 8))
top_missing = missing_pd.head(20).sort_values("null_pct", ascending=True)
plt.barh(top_missing["column"], top_missing["null_pct"])
plt.title("Top 20 Columns by Missing Percentage")
plt.xlabel("Null percentage")
plt.ylabel("Column")
save_current_chart("eda_missing_top20.png")
plt.show()

In [ ]:
# =========================
# 17. FILE SIZE EDA
# =========================

file_pd = (
    features_df
    .select(
        (F.col("file_size_bytes") / 1024).alias("file_size_kb"),
        (F.col("file_size_bytes") / (1024 * 1024)).alias("file_size_mb")
    )
    .where(F.col("file_size_kb").isNotNull())
    .sample(False, 0.5, 42)
    .toPandas()
)

plt.figure(figsize=(10, 5))
plt.hist(file_pd["file_size_kb"], bins=60)
plt.title("File Size Distribution (KB)")
plt.xlabel("File size (KB)")
plt.ylabel("Frequency")
save_current_chart("eda_file_size_kb_hist.png")
plt.show()

plt.figure(figsize=(10, 5))
plt.boxplot(file_pd["file_size_mb"].dropna(), vert=False)
plt.title("File Size Boxplot (MB)")
plt.xlabel("MB")
save_current_chart("eda_file_size_mb_box.png")
plt.show()

In [ ]:
# =========================
# 18. RESOLUTION + ASPECT RATIO EDA
# =========================

res_pd = (
    features_df
    .withColumn("megapixels", (F.col("width") * F.col("height")) / 1000000.0)
    .select("width", "height", "megapixels", "aspect_ratio", "orientation")
    .where(F.col("width").isNotNull() & F.col("height").isNotNull())
    .sample(False, 0.5, 42)
    .toPandas()
)

print("Resolution rows used:", len(res_pd))

plt.figure(figsize=(7, 7))
plt.scatter(res_pd["width"], res_pd["height"], s=8, alpha=0.35)
plt.title("Image Resolution Scatter")
plt.xlabel("Width")
plt.ylabel("Height")
save_current_chart("eda_resolution_scatter.png")
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(res_pd["aspect_ratio"].dropna(), bins=40)
plt.title("Aspect Ratio Distribution")
plt.xlabel("Width / Height")
plt.ylabel("Frequency")
save_current_chart("eda_aspect_ratio_hist.png")
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(res_pd["megapixels"].dropna(), bins=50)
plt.title("Megapixel Distribution")
plt.xlabel("Megapixels")
plt.ylabel("Frequency")
save_current_chart("eda_megapixel_hist.png")
plt.show()

In [ ]:
# =========================
# 19. ORIENTATION / ASPECT CATEGORY
# =========================

aspect_bucket_df = (
    features_df
    .withColumn(
        "aspect_bucket",
        F.when(F.col("aspect_ratio").isNull(), "unknown")
         .when(F.col("aspect_ratio") < 0.9, "tall")
         .when((F.col("aspect_ratio") >= 0.9) & (F.col("aspect_ratio") <= 1.1), "square-ish")
         .when((F.col("aspect_ratio") > 1.1) & (F.col("aspect_ratio") <= 1.8), "standard-wide")
         .otherwise("ultra-wide")
    )
)

aspect_pd = (
    aspect_bucket_df.groupBy("aspect_bucket")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

plt.figure(figsize=(8, 4))
plt.bar(aspect_pd["aspect_bucket"], aspect_pd["count"])
plt.title("Aspect Ratio Categories")
plt.xlabel("Aspect bucket")
plt.ylabel("Count")
save_current_chart("eda_aspect_bucket.png")
plt.show()

In [ ]:
# =========================
# 20. EXIF COVERAGE AND CAMERA ANALYSIS
# =========================

exif_cov = features_df.select(
    F.sum(F.col("capture_ts").isNotNull().cast("int")).alias("capture_ts"),
    F.sum(F.col("camera_make").isNotNull().cast("int")).alias("camera_make"),
    F.sum(F.col("camera_model").isNotNull().cast("int")).alias("camera_model"),
    F.sum(F.col("gps_lat").isNotNull().cast("int")).alias("gps_lat"),
    F.sum(F.col("gps_lon").isNotNull().cast("int")).alias("gps_lon"),
    F.sum(F.col("iso").isNotNull().cast("int")).alias("iso"),
    F.sum(F.col("exposure_time").isNotNull().cast("int")).alias("exposure_time"),
    F.sum(F.col("f_number").isNotNull().cast("int")).alias("f_number"),
    F.sum(F.col("focal_length").isNotNull().cast("int")).alias("focal_length")
).toPandas().T.reset_index()

exif_cov.columns = ["field", "count"]
exif_cov

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(exif_cov["field"], exif_cov["count"])
plt.title("EXIF Field Coverage")
plt.xlabel("EXIF field")
plt.ylabel("Count")
plt.xticks(rotation=30)
save_current_chart("eda_exif_field_coverage.png")
plt.show()

camera_make_pd = (
    features_df.where(F.col("camera_make").isNotNull())
    .groupBy("camera_make")
    .count()
    .orderBy(F.desc("count"))
    .limit(TOP_N_CAMERAS)
    .toPandas()
)

if len(camera_make_pd) > 0:
    plt.figure(figsize=(12, 6))
    plt.barh(camera_make_pd["camera_make"][::-1], camera_make_pd["count"][::-1])
    plt.title("Top Camera Makes")
    plt.xlabel("Count")
    save_current_chart("eda_top_camera_makes.png")
    plt.show()

camera_model_pd = (
    features_df.where(F.col("camera_model").isNotNull())
    .groupBy("camera_model")
    .count()
    .orderBy(F.desc("count"))
    .limit(TOP_N_CAMERAS)
    .toPandas()
)

if len(camera_model_pd) > 0:
    plt.figure(figsize=(12, 6))
    plt.barh(camera_model_pd["camera_model"][::-1], camera_model_pd["count"][::-1])
    plt.title("Top Camera Models")
    plt.xlabel("Count")
    save_current_chart("eda_top_camera_models.png")
    plt.show()

In [ ]:
# =========================
# 21. TAG EDA
# =========================

tag_overview_pd = features_df.select(
    F.count("*").alias("n_images"),
    F.sum(F.col("has_tags").cast("int")).alias("with_tags"),
    F.avg(F.col("unique_tag_count")).alias("avg_unique_tag_count"),
    F.expr("percentile_approx(unique_tag_count, 0.5)").alias("median_unique_tag_count"),
    F.max("unique_tag_count").alias("max_unique_tag_count")
).toPandas()

tag_overview_pd

In [ ]:
tag_count_pd = (
    features_df.select("unique_tag_count")
    .where(F.col("unique_tag_count").isNotNull())
    .toPandas()
)

plt.figure(figsize=(10, 5))
plt.hist(tag_count_pd["unique_tag_count"], bins=40)
plt.title("Unique Tag Count per Image")
plt.xlabel("Unique tag count")
plt.ylabel("Frequency")
save_current_chart("eda_unique_tag_count_hist.png")
plt.show()

top_tags_pd = (
    features_df
    .select(F.explode_outer("tags").alias("tag"))
    .where(F.col("tag").isNotNull())
    .groupBy("tag")
    .count()
    .orderBy(F.desc("count"))
    .limit(TOP_N_TAGS)
    .toPandas()
)

plt.figure(figsize=(12, 10))
plt.barh(top_tags_pd["tag"][::-1], top_tags_pd["count"][::-1])
plt.title(f"Top {TOP_N_TAGS} Tags")
plt.xlabel("Count")
plt.ylabel("Tag")
save_current_chart("eda_top_tags.png")
plt.show()

top_tags_pd.head(20)

In [ ]:
# =========================
# 22. TAG LONG-TAIL + CO-OCCURRENCE
# =========================

tag_freq_df = (
    features_df
    .select(F.explode_outer("tags").alias("tag"))
    .where(F.col("tag").isNotNull())
    .groupBy("tag")
    .count()
)

tail_pd = tag_freq_df.select(
    F.count("*").alias("n_unique_tags"),
    F.sum((F.col("count") == 1).cast("int")).alias("tags_once"),
    F.sum((F.col("count") <= 5).cast("int")).alias("tags_le_5"),
    F.sum((F.col("count") <= 10).cast("int")).alias("tags_le_10")
).toPandas()

tail_pd

In [ ]:
selected_tags = [t for t in top_tags_pd["tag"].head(12).tolist()]

co_pd = (
    features_df
    .select("photo_id", F.explode_outer("tags").alias("tag"))
    .where(F.col("tag").isin(selected_tags))
    .toPandas()
)

mat = pd.DataFrame(0, index=selected_tags, columns=selected_tags)
pairs = co_pd.groupby("photo_id")["tag"].apply(list)

for arr in pairs:
    arr = sorted(set(arr))
    for i in range(len(arr)):
        for j in range(len(arr)):
            mat.loc[arr[i], arr[j]] += 1

plt.figure(figsize=(10, 8))
plt.imshow(mat.values)
plt.xticks(range(len(selected_tags)), selected_tags, rotation=90)
plt.yticks(range(len(selected_tags)), selected_tags)
plt.title("Tag Co-occurrence Matrix")
plt.colorbar()
save_current_chart("eda_tag_cooccurrence.png")
plt.show()

In [ ]:
# =========================
# 23. TEMPORAL EDA
# =========================

time_cov = features_df.select(
    F.sum(F.col("year").isNotNull().cast("int")).alias("with_year"),
    F.sum(F.col("month").isNotNull().cast("int")).alias("with_month"),
    F.sum(F.col("day").isNotNull().cast("int")).alias("with_day")
).toPandas()

time_cov

In [ ]:
year_pd = (
    features_df.where(F.col("year").isNotNull())
    .groupBy("year")
    .count()
    .orderBy("year")
    .toPandas()
)

if len(year_pd) > 0:
    plt.figure(figsize=(11, 5))
    plt.plot(year_pd["year"], year_pd["count"], marker="o")
    plt.title("Image Count by Year")
    plt.xlabel("Year")
    plt.ylabel("Count")
    save_current_chart("eda_year_trend.png")
    plt.show()

month_pd = (
    features_df.where(F.col("month").isNotNull())
    .groupBy("month")
    .count()
    .orderBy("month")
    .toPandas()
)

if len(month_pd) > 0:
    plt.figure(figsize=(9, 4))
    plt.bar(month_pd["month"], month_pd["count"])
    plt.title("Image Count by Month")
    plt.xlabel("Month")
    plt.ylabel("Count")
    save_current_chart("eda_month_distribution.png")
    plt.show()

In [ ]:
time_detail_df = (
    features_df
    .where(F.col("capture_ts").isNotNull())
    .withColumn("dow", F.date_format("capture_ts", "E"))
    .withColumn("hour", F.hour("capture_ts"))
)

dow_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_pd = time_detail_df.groupBy("dow").count().toPandas()

if len(dow_pd) > 0:
    dow_pd["dow"] = pd.Categorical(dow_pd["dow"], categories=dow_order, ordered=True)
    dow_pd = dow_pd.sort_values("dow")

    plt.figure(figsize=(8, 4))
    plt.bar(dow_pd["dow"], dow_pd["count"])
    plt.title("Image Count by Day of Week")
    plt.xlabel("Day")
    plt.ylabel("Count")
    save_current_chart("eda_day_of_week.png")
    plt.show()

hour_pd = time_detail_df.groupBy("hour").count().orderBy("hour").toPandas()
if len(hour_pd) > 0:
    plt.figure(figsize=(10, 4))
    plt.bar(hour_pd["hour"], hour_pd["count"])
    plt.title("Image Count by Hour of Day")
    plt.xlabel("Hour")
    plt.ylabel("Count")
    save_current_chart("eda_hour_distribution.png")
    plt.show()

In [ ]:
# =========================
# 24. LOCATION EDA
# =========================

location_cov_pd = features_df.select(
    F.sum(F.col("has_gps_exif").cast("int")).alias("with_gps_exif"),
    F.sum(F.col("has_location_tag").cast("int")).alias("with_location_tag"),
    F.sum((F.col("has_gps_exif") | F.col("has_location_tag")).cast("int")).alias("with_any_location_signal")
).toPandas()

location_cov_pd

In [ ]:
top_locations_pd = (
    features_df
    .select(F.explode_outer("location_candidates").alias("location"))
    .where(F.col("location").isNotNull())
    .groupBy("location")
    .count()
    .orderBy(F.desc("count"))
    .limit(TOP_N_LOCATIONS)
    .toPandas()
)

plt.figure(figsize=(12, 9))
plt.barh(top_locations_pd["location"][::-1], top_locations_pd["count"][::-1])
plt.title("Top Extracted Locations from Tags")
plt.xlabel("Count")
plt.ylabel("Location")
save_current_chart("eda_top_locations.png")
plt.show()

top_locations_pd.head(20)

In [ ]:
# =========================
# 25. SEMANTIC BUCKET EDA
# =========================

bucket_pd = (
    features_df.groupBy("semantic_bucket")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

plt.figure(figsize=(10, 5))
plt.bar(bucket_pd["semantic_bucket"], bucket_pd["count"])
plt.title("Semantic Bucket Distribution")
plt.xlabel("Bucket")
plt.ylabel("Count")
plt.xticks(rotation=25)
save_current_chart("eda_semantic_buckets.png")
plt.show()

bucket_pd

In [ ]:
# =========================
# 26. MEMORY-CANDIDATE EDA
# =========================

story_eda_df = (
    features_df
    .withColumn(
        "memory_location",
        F.coalesce(
            F.col("primary_location_tag"),
            F.when(
                F.col("gps_lat").isNotNull() & F.col("gps_lon").isNotNull(),
                F.concat(F.lit("gps:"), F.round("gps_lat", 2).cast("string"), F.lit(","), F.round("gps_lon", 2).cast("string"))
            )
        )
    )
    .withColumn("memory_month", F.when(F.col("capture_ts").isNotNull(), F.date_format("capture_ts", "yyyy-MM")))
    .withColumn(
        "memory_candidate",
        F.col("memory_location").isNotNull() &
        F.col("memory_month").isNotNull() &
        F.col("semantic_bucket").isNotNull() &
        (F.col("semantic_bucket") != "unknown")
    )
)

story_eda_df = story_eda_df.cache()
_ = story_eda_df.count()

memory_cov_pd = story_eda_df.select(
    F.count("*").alias("total"),
    F.sum(F.col("memory_candidate").cast("int")).alias("memory_candidates")
).toPandas()

memory_cov_pd

In [ ]:
top_memory_clusters_pd = (
    story_eda_df
    .where(F.col("memory_candidate"))
    .groupBy("memory_month", "memory_location", "semantic_bucket")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
    .toPandas()
)

top_memory_clusters_pd.head(20)

In [ ]:
if len(top_memory_clusters_pd) > 0:
    plt.figure(figsize=(12, 8))
    labels = [
        f"{r['memory_month']} | {r['memory_location']} | {r['semantic_bucket']}"
        for _, r in top_memory_clusters_pd.head(15).iterrows()
    ]
    vals = top_memory_clusters_pd.head(15)["count"].tolist()
    plt.barh(labels[::-1], vals[::-1])
    plt.title("Top Memory Clusters")
    plt.xlabel("Image count")
    plt.ylabel("Cluster")
    save_current_chart("eda_top_memory_clusters.png")
    plt.show()

In [ ]:
# =========================
# 27. IMAGE GRID HELPERS
# =========================

def load_and_prepare_image(path_str, thumb_size=(120, 120)):
    try:
        local_path = spark_file_uri_to_local_path(path_str)
        with Image.open(local_path) as img:
            img = img.convert("RGB")
            img.thumbnail(thumb_size)
            canvas = Image.new("RGB", thumb_size, (245, 245, 245))
            x = (thumb_size[0] - img.size[0]) // 2
            y = (thumb_size[1] - img.size[1]) // 2
            canvas.paste(img, (x, y))
            return canvas
    except Exception:
        fallback = Image.new("RGB", thumb_size, (230, 230, 230))
        draw = ImageDraw.Draw(fallback)
        draw.text((10, 50), "ERR", fill=(80, 80, 80))
        return fallback

def show_image_grid(pdf, title, rows=10, cols=10, thumb_size=(120, 120)):
    n = min(len(pdf), rows * cols)
    fig, axes = plt.subplots(rows, cols, figsize=(18, 18))
    axes = axes.flatten()

    for i in range(rows * cols):
        ax = axes[i]
        ax.axis("off")
        if i < n:
            row = pdf.iloc[i]
            img = load_and_prepare_image(row["image_path"], thumb_size=thumb_size)
            ax.imshow(img)
            pid = row["photo_id"] if "photo_id" in pdf.columns else ""
            ax.set_title(f"id={pid}", fontsize=7)

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
# =========================
# 28. 10x10 LOCATION GRID
# =========================

TARGET_LOCATION = "London"   # change this based on top_locations_pd

location_grid_pd = (
    features_df
    .where(F.array_contains(F.col("location_candidates"), TARGET_LOCATION))
    .orderBy(F.rand(RANDOM_SEED))
    .select("photo_id", "image_path", "primary_location_tag", "capture_ts")
    .limit(100)
    .toPandas()
)

print(f"Images found for {TARGET_LOCATION}: {len(location_grid_pd)}")
location_grid_pd.head()

In [ ]:
if len(location_grid_pd) > 0:
    show_image_grid(location_grid_pd, f"10x10 Grid for Location: {TARGET_LOCATION}", rows=10, cols=10)

In [ ]:
# =========================
# 29. 10x10 SEMANTIC BUCKET GRID
# =========================

TARGET_BUCKET = "landscape_nature"

bucket_grid_pd = (
    features_df
    .where(F.col("semantic_bucket") == TARGET_BUCKET)
    .orderBy(F.rand(RANDOM_SEED))
    .select("photo_id", "image_path", "semantic_bucket", "capture_ts")
    .limit(100)
    .toPandas()
)

print(f"Images found for bucket {TARGET_BUCKET}: {len(bucket_grid_pd)}")
bucket_grid_pd.head()

In [ ]:
if len(bucket_grid_pd) > 0:
    show_image_grid(bucket_grid_pd, f"10x10 Grid for Semantic Bucket: {TARGET_BUCKET}", rows=10, cols=10)

In [ ]:
# =========================
# 30. MEMORY-STYLE LOCATION + MONTH GRID
# =========================

TARGET_LOCATION = "London"
TARGET_MONTH = "2008-07"

memory_grid_pd = (
    story_eda_df
    .where(F.col("memory_candidate"))
    .where(F.col("memory_location") == TARGET_LOCATION)
    .where(F.col("memory_month") == TARGET_MONTH)
    .orderBy(F.rand(RANDOM_SEED))
    .select("photo_id", "image_path", "memory_location", "memory_month", "semantic_bucket")
    .limit(100)
    .toPandas()
)

print(f"Images found for {TARGET_LOCATION} in {TARGET_MONTH}: {len(memory_grid_pd)}")
memory_grid_pd.head()

In [ ]:
if len(memory_grid_pd) > 0:
    show_image_grid(memory_grid_pd, f"Memory-style Grid: {TARGET_LOCATION} | {TARGET_MONTH}", rows=10, cols=10)

## Recommended presentation visuals

Use these 8–10 visuals:

1. missingness / metadata coverage
2. file size distribution
3. resolution scatter
4. aspect ratio distribution
5. EXIF field coverage
6. top tags
7. top extracted locations
8. year/month trend
9. semantic bucket distribution
10. top memory clusters
11. 10x10 location grid
12. 10x10 memory-style grid